# Vision Transformer  


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Config

In [ ]:

d_model = 128
n_heads = 4
num_layers = 4
d_ff = 256
num_classes = 10
batch_size = 64
epochs = 5
lr = 1e-3

## Data

In [ ]:

transform = transforms.ToTensor()
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## CNN Feature Extractor

In [ ]:

cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),   # 28→14

    nn.Conv2d(32, 64, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)    # 14→7
).to(device)

## Parameters (Patch Embed, CLS Token, Pos Embed)

In [ ]:

patch_embed = nn.Linear(64, d_model).to(device)

cls_token = nn.Parameter(torch.randn(1,1,d_model)).to(device)
pos_embed = nn.Parameter(torch.randn(1, 49+1, d_model)).to(device)

## Attention

In [ ]:

def scaled_dot_product(q,k,v):
    dk = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(dk)
    attn = torch.softmax(scores, dim=-1)
    out = torch.matmul(attn, v)
    return out, attn

def multi_head_attention(x, Wq, Wk, Wv, Wo):
    B, T, D = x.shape

    q = Wq(x)
    k = Wk(x)
    v = Wv(x)

    q = q.view(B, T, n_heads, D//n_heads).transpose(1,2)
    k = k.view(B, T, n_heads, D//n_heads).transpose(1,2)
    v = v.view(B, T, n_heads, D//n_heads).transpose(1,2)

    out, attn = scaled_dot_product(q,k,v)

    out = out.transpose(1,2).contiguous().view(B,T,D)
    return Wo(out), attn

## Feed Forward

In [ ]:

def feed_forward(x, W1, W2):
    return W2(F.relu(W1(x)))

## Build Transformer Layers

In [ ]:

def build_layer():
    return {
        "attn": [nn.Linear(d_model,d_model).to(device) for _ in range(4)],
        "ff": [nn.Linear(d_model,d_ff).to(device), nn.Linear(d_ff,d_model).to(device)],
        "ln1": nn.LayerNorm(d_model).to(device),
        "ln2": nn.LayerNorm(d_model).to(device)
    }

layers = [build_layer() for _ in range(num_layers)]

mlp_head = nn.Linear(d_model, num_classes).to(device)

## Forward Pass

In [ ]:

def forward(x):
    x = x.to(device)


    x = cnn(x)  # (B,64,7,7)
    B, C, H, W = x.shape
    x = x.reshape(B, C, H*W).permute(0,2,1)  

    x = patch_embed(x)

    cls = cls_token.expand(B, -1, -1)
    x = torch.cat([cls, x], dim=1)

    x = x + pos_embed

    attn_maps = []

    for layer in layers:
        attn_out, attn = multi_head_attention(x, *layer["attn"])
        attn_maps.append(attn)

        x = layer["ln1"](x + attn_out)

        ff = feed_forward(x, *layer["ff"])
        x = layer["ln2"](x + ff)

    return mlp_head(x[:,0]), attn_maps

## Optimizer & Loss

In [ ]:

params = list(cnn.parameters()) + list(patch_embed.parameters()) + list(mlp_head.parameters()) + [cls_token, pos_embed]

for layer in layers:
    for v in layer.values():
        if isinstance(v, list):
            for m in v:
                params += list(m.parameters())
        else:
            params += list(v.parameters())

optimizer = torch.optim.Adam(params, lr=lr)
loss_fn = nn.CrossEntropyLoss()

## Training Loop

In [ ]:
for epoch in range(epochs):
    for x, y in loader:
        y = y.to(device)

        logits, _ = forward(x)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

## Attention Visualisation

In [ ]:

def visualize_attention(attn_maps):
    attn = attn_maps[-1][0,0].detach().cpu()

    plt.imshow(attn, cmap='viridis')
    plt.title("Attention Map")
    plt.colorbar()
    plt.show()

## Inference

In [ ]:

def predict(img):
    logits, attn_maps = forward(img.unsqueeze(0))
    pred = torch.argmax(logits, dim=1).item()

    print("Predicted digit:", pred)
    visualize_attention(attn_maps)

## Test

In [ ]:

x, y = next(iter(loader))
predict(x[0])